In [9]:
pip install keybert

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
pip install sentence-transformers numpy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Load data


In [12]:
import json

# Load data from file
with open("products.json", "r", encoding="utf-8") as f:
    products = json.load(f)
    
print(f"Total products loaded: {len(products)}")

# Show first 2 products
for p in products[:]:
    print("\n--- SAMPLE PRODUCT ---")
    print(json.dumps(p, indent=2))

Total products loaded: 110

--- SAMPLE PRODUCT ---
{
  "id": "cml23srgd00bima01wk8mku8u",
  "shopId": "cml21tavh00a7ma018cjo99db",
  "categoryId": "cml0ug1d6003to201xuwjkmcn",
  "name": "Gift Wrapping",
  "slug": "gift-wrapping-The Gift Atelier",
  "description": "Our Gift Wrapping service adds the perfect finishing touch to every present. Using premium materials and refined details, each gift is carefully wrapped to reflect thoughtfulness, style, and care.",
  "shortDescription": "Our Gift Accessories are designed to elevate every gift with thoughtful detail and refined style. From elegant wrapping and premium boxes to ribbons, ",
  "mainImage": "https://s3aws.emilo.in/products/1769851348474-322c67f6-af06-4ab9-b9d4-4205721d8b0a.png",
  "brand": "The Gift Atelier",
  "sku": "BR-ML23SN9V-4MQB",
  "price": "499",
  "discountPercentage": "10",
  "stock": 148,
  "lowStockThreshold": 20,
  "weight": "100",
  "dimensions": "{\"length\":30,\"width\":20,\"height\":30}",
  "isApproved": true,
 

Tags creations

In [13]:
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import numpy as np
import json

# -----------------------------
# Load models
# -----------------------------
kw_model = KeyBERT()
model = SentenceTransformer('all-MiniLM-L6-v2')

# -----------------------------
# Load data
# -----------------------------
with open("products.json", "r", encoding="utf-8") as f:
    products = json.load(f)

# -----------------------------
# STEP 1: Build texts
# -----------------------------
def build_text(p):
    return f"{p['name']} {p['description']} {p['shortDescription']}".lower()

texts = [build_text(p) for p in products]


# -----------------------------
# STEP 2: Extract keywords
# -----------------------------
all_keywords = []

for text in texts:
    keywords = kw_model.extract_keywords(text, top_n=5)

    for kw, score in keywords:
        all_keywords.append(kw.lower())

# unique tags
tags = list(set(all_keywords))

print(f"\nRaw tags: {len(tags)}")


# -----------------------------
# STEP 3: Clean tags
# -----------------------------
def clean_tags(tags):
    cleaned = []

    for t in tags:
        t = t.strip()

        if len(t) < 3:
            continue

        if len(t.split()) > 3:
            continue

        cleaned.append(t)

    return list(set(cleaned))


cleaned_tags = clean_tags(tags)

print(f"Cleaned tags: {len(cleaned_tags)}")


# -----------------------------
# STEP 4: Merge similar tags
# -----------------------------
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def merge_similar_tags(tags, threshold=0.8):
    merged = []

    # compute embeddings once
    tag_embeddings = model.encode(tags)

    for i, tag in enumerate(tags):
        tag_emb = tag_embeddings[i]

        found = False

        for j, m in enumerate(merged):
            m_emb = model.encode(m)

            sim = cosine_similarity(tag_emb, m_emb)

            if sim > threshold:
                found = True
                break

        if not found:
            merged.append(tag)

    return merged


merged_tags = merge_similar_tags(cleaned_tags)

print(f"Merged tags: {len(merged_tags)}")





Raw tags: 283
Cleaned tags: 281
Merged tags: 236


In [14]:
import json

with open("merged_tags.json", "w", encoding="utf-8") as f:
    json.dump(merged_tags, f, indent=2)

print("✅ Tags saved to merged_tags.json")

with open("merged_tags.json", "r", encoding="utf-8") as f:
    merged_tags = json.load(f)

print(merged_tags)

✅ Tags saved to merged_tags.json
['wax', 'studs', 'favorite', 'chocolates', 'mental', 'infusions', 'bike', 'festive', 'suit', 'oils', 'digital', 'polo', 'sensual', 'metal', 'fragrance', 'blossom', 'recycled', 'florists', 'instruments', 'kashmiri', 'toys', 'lilies', 'quailty', 'polished', 'accessory', 'stacked', 'tulips', 'straigh', 'bags', 'cuddly', 'organic', 'touch', 'rose', 'bears', 'creamy', 'newborn', 'fountain', 'traditional', 'drop', 'engagements', 'bamboo', 'aromas', 'good', 'wall', 'sarees', 'gifting', 'card', 'pouches', 'handcrafted', 'indoor', 'seal', 'clothes', 'pen', 'upcycled', 'bold', 'arts', 'multicolor', 'keychains', 'wellness', 'care', 'floral', 'pearls', 'hearts', 'blooms', 'cleanse', 'office', 'sky', 'attire', 'description', 'dark', 'designed', 'playful', 'gifted', 'luxury', 'routine', 'seasonal', 'delicate', 'pearl', 'open', 'psychological', 'cherry', 'kurtis', 'cargo', 'plush', 'elegance', 'lifestyle', 'music', 'romance', 'shirt', 'décor', 'skin', 'denim', 'premiu

occasion_tags

In [15]:

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

with open("merged_tags.json", "r", encoding="utf-8") as f:
    merged_tags = json.load(f)



TAGS = merged_tags


#Precompute Tag Embeddings
tag_embeddings = {
    tag: model.encode(tag)
    for tag in TAGS
}

#Cosine Similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


#AI Tag Generator Function
def generate_ai_tags(text, threshold=0.4):
    product_embedding = model.encode(text)

    assigned_tags = []

    for tag, tag_emb in tag_embeddings.items():
        sim = cosine_similarity(product_embedding, tag_emb)

        if sim >= threshold:
            assigned_tags.append(tag)

    return assigned_tags

Is Valid Product

In [16]:
def is_valid_product(p):
    return (
        p.get("status") == "ACTIVE" and
        p.get("isApproved") == True and
        p.get("stock", 0) > 0
    )

Get Price Bucket

In [17]:
def get_price_bucket(price):
    try:
        price = float(price)
    except:
        return "unknown"

    if price < 500:
        return "low"
    elif price < 2000:
        return "medium"
    else:
        return "high"

Final data creations

In [18]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

with open("merged_tags.json", "r", encoding="utf-8") as f:
    merged_tags = json.load(f)

processed_products = []

for p in products:
    text = build_text(p)
    occasion_tags = generate_ai_tags(text )
    valid = is_valid_product(p)
    price_bucket = get_price_bucket(p.get("price", 0))

    new_product = {
        **p,
        "searchText": text,
        "tags": merged_tags,
        "occasionTags": occasion_tags,
        "isValid": valid,
        "priceBucket": price_bucket
    }

    processed_products.append(new_product)
    import json

with open("final.json", "w", encoding="utf-8") as f:
    json.dump(processed_products, f, indent=2)

print("✅ Tags saved to final.json")

✅ Tags saved to final.json
